In [1]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [13]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load the NorBERT model and tokenizer
model_name = "ltgoslo/norbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Read the content of the text document
with open("document.txt", "r", encoding="utf-8") as file:
    document = file.read()

# Tokenize the document
tokens = tokenizer.tokenize(document)
max_length = 512 - 2  # Account for special tokens [CLS] and [SEP]

# Split tokens into chunks of max_length
chunks = [tokens[i:i + max_length] for i in range(0, len(tokens), max_length)]

# Function to add special tokens to each chunk
def add_special_tokens(chunk):
    return [tokenizer.cls_token] + chunk + [tokenizer.sep_token]

# Process each chunk through the model and collect embeddings
embeddings = []

for chunk in chunks:
    chunk_with_special_tokens = add_special_tokens(chunk)
    inputs = tokenizer(chunk_with_special_tokens, return_tensors="pt", is_split_into_words=True, padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        chunk_embeddings = outputs.last_hidden_state
        embeddings.append(chunk_embeddings)

# Function to combine embeddings later
def combine_embeddings(embeddings, method="average"):
    """
    Combine embeddings using the specified method.
    
    :param embeddings: List of chunk embeddings
    :param method: Method to combine embeddings. Options: "average", "sum", "concat"
    :return: Combined embeddings
    """
    # Concatenate all chunk embeddings along the sequence dimension
    embeddings = torch.cat(embeddings, dim=1)

    if method == "average":
        # Average the embeddings across the token dimension
        combined_embedding = torch.mean(embeddings, dim=1)
    elif method == "sum":
        # Sum the embeddings across the token dimension
        combined_embedding = torch.sum(embeddings, dim=1)
    elif method == "concat":
        # Concatenate embeddings along the batch dimension
        combined_embedding = embeddings.view(embeddings.size(0), -1)
    else:
        raise ValueError("Invalid method for combining embeddings. Choose from 'average', 'sum', 'concat'.")

    return combined_embedding

# Combine the embeddings using the desired method
document_embedding = combine_embeddings(embeddings, method="average")

# Print the resulting embedding
print(document_embedding)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


RuntimeError: The size of tensor a (866) must match the size of tensor b (512) at non-singleton dimension 1